# 02 — Preprocessing & Feature Engineering
 (Data Engineer)

This notebook demonstrates the preprocessing pipeline defined in
`src/preprocessing.py` and `src/features.py`. It:

1. Loads and cleans the raw data
2. Applies feature engineering
3. Splits into train/test (stratified on `job_ready`) **before** fitting
   any transformer, to prevent data leakage
4. Fits the `ColumnTransformer` on the training set only
5. Saves the fitted pipeline and processed data for the modeling notebooks

In [ ]:
import sys
sys.path.append("../src")

from preprocessing import (
    load_raw_data, clean_data, run_preprocessing, NUMERIC_FEATURES, CATEGORICAL_FEATURES
)
from features import add_engineered_features, ENGINEERED_FEATURE_NAMES

df = load_raw_data("../data/raw/student_employability.csv")
print(df.shape)
df.head(3)

## Step 1: Clean the data

In [ ]:
df_clean = clean_data(df)
print(f"Rows before: {len(df)}, after cleaning: {len(df_clean)}")

## Step 2: Feature engineering

In [ ]:
df_engineered = add_engineered_features(df_clean)
print("New engineered columns:")
for c in ENGINEERED_FEATURE_NAMES:
    print(" -", c)
df_engineered[ENGINEERED_FEATURE_NAMES].describe().T

## Step 3: Why data leakage prevention matters here

If we fit the scaler/imputer on the FULL dataset (train + test) before
splitting, information from the test set (e.g. its mean/median values)
would leak into the training process, making our evaluation metrics
overly optimistic. `run_preprocessing()` in `src/preprocessing.py` avoids
this by splitting first, then calling `.fit()` only on `X_train`.

In [ ]:
(X_train, X_test, y_job_train, y_job_test,
 y_track_train, y_track_test, preprocessor) = run_preprocessing()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain job_ready balance:")
print(y_job_train.value_counts(normalize=True))
print("\nTest job_ready balance:")
print(y_job_test.value_counts(normalize=True))

## Step 4: Inspect the fitted pipeline

The pipeline has two branches:
- **num**: median imputation + standard scaling for all numeric features
  (including the engineered ratio/index features)
- **cat**: most-frequent imputation + one-hot encoding for `degree`

In [ ]:
preprocessor

In [ ]:
X_train_transformed = preprocessor.transform(X_train)
print("Transformed shape:", X_train_transformed.shape)
print("Any NaNs after transform?", pd_isna_check := __import__("numpy").isnan(X_train_transformed).any())

**Result:** stratified 80/20 split, no NaNs remain after transformation,
and `data/processed/processed_data.csv` + `models/preprocessing_pipeline.joblib`
are saved for the training notebooks.

**Next notebook:** `03_baseline_models.ipynb` (Dushant).